<a href="https://colab.research.google.com/github/DiyaMalik07/SC_Project/blob/main/SGA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pymoo

import numpy as np
import pandas as pd
import time
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.operators.crossover.hux import HalfUniformCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.optimize import minimize

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Libraries imported successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 1.7 MB/s eta 0:00:00
Libraries imported successfully


In [2]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/internet_ads/ad.data"
urllib.request.urlretrieve(url, "ad.data")

df = pd.read_csv("ad.data", header=None, low_memory=False)
print(f"Dataset shape (rows, columns): {df.shape}")

print("Preprocessing data")
df = df.replace(r'^\s*\?\s*$', np.nan, regex=True)

label_col = df.columns[-1]
y = df[label_col].astype(str).str.strip().apply(
    lambda v: 1 if v.lower().startswith('ad') else 0
).values

X = df.drop(columns=[label_col])
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.mean())
X = X.values

n_features = X.shape[1]
print(f"Total features: {n_features}")
print(f"Total instances: {X.shape[0]}")
print(f"Class balance -> ad: {sum(y==1)}  |  nonad: {sum(y==0)}")

Dataset shape (rows, columns): (3279, 1559)
Preprocessing data
Total features: 1558
Total instances: 3279
Class balance -> ad: 459  |  nonad: 2820


In [3]:
print("Splitting data (Train 60% / Validation 20% / Test 20%)")

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Train samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

Splitting data (Train 60% / Validation 20% / Test 20%)
Train samples: 1967
Validation samples: 656
Test samples: 656


In [4]:
print("Random Forest on ALL features")

start_time = time.time()
baseline_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
baseline_model.fit(X_train, y_train)
baseline_train_time = time.time() - start_time

baseline_predictions = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_f1 = f1_score(y_test, baseline_predictions)

print(f"Baseline Accuracy : {baseline_accuracy:.4f}")
print(f"Baseline F1-score : {baseline_f1:.4f}")
print(f"Features used     : {n_features}")
print(f"Training time     : {baseline_train_time:.2f} seconds")

Random Forest on ALL features
Baseline Accuracy : 0.9665
Baseline F1-score : 0.8690
Features used     : 1558
Training time     : 1.52 seconds


In [5]:
print("Setting up SGA feature selection")

class DiverseBinarySampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        population = np.zeros((n_samples, problem.n_var), dtype=bool)
        for i in range(n_samples):
            selection_prob = np.random.uniform(0.02, 0.80)
            population[i] = np.random.rand(problem.n_var) < selection_prob
            if population[i].sum() == 0:
                population[i, np.random.randint(0, problem.n_var)] = True
        return population

class SGAFeatureSelectionProblem(Problem):
    def __init__(self, X_train, y_train, X_val, y_val, alpha=0.95):
        super().__init__(n_var=n_features, n_obj=1, n_constr=0, xl=0, xu=1, vtype=bool)
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.alpha = alpha

    def _evaluate(self, population, out, *args, **kwargs):
        fitness_list = []

        for individual in population:
            mask = individual.astype(bool)

            if mask.sum() == 0:
                mask[np.random.randint(0, self.n_var)] = True

            model = RandomForestClassifier(n_estimators=30, random_state=RANDOM_SEED, n_jobs=-1)
            model.fit(self.X_train[:, mask], self.y_train)
            predictions = model.predict(self.X_val[:, mask])

            f1 = f1_score(self.y_val, predictions)
            f1_loss = 1.0 - f1
            feature_fraction = mask.sum() / self.n_var

            fitness = (self.alpha * f1_loss) + ((1.0 - self.alpha) * feature_fraction)
            fitness_list.append(fitness)

        out["F"] = np.column_stack([fitness_list])

sga_problem = SGAFeatureSelectionProblem(X_train, y_train, X_val, y_val, alpha=0.95)

Setting up SGA feature selection


In [6]:
POPULATION_SIZE = 40
GENERATIONS = 30

sga_algorithm = GA(
    pop_size=POPULATION_SIZE,
    sampling=DiverseBinarySampling(),
    crossover=HalfUniformCrossover(),
    mutation=BitflipMutation(prob=5.0 / n_features),
    eliminate_duplicates=True
)

print(f"Running SGA optimization for {GENERATIONS} generations")
start_time = time.time()

sga_result = minimize(
    sga_problem,
    sga_algorithm,
    termination=('n_gen', GENERATIONS),
    seed=RANDOM_SEED,
    verbose=True
)

sga_run_time = time.time() - start_time
print(f"\nSGA finished in {sga_run_time:.2f} seconds")

Running SGA optimization for 30 generations
n_gen  |  n_eval  |     f_avg     |     f_min    
     1 |       40 |  0.1582475666 |  0.0722278598
     2 |       80 |  0.0951430745 |  0.0722278598
     3 |      120 |  0.0848879336 |  0.0691477678
     4 |      160 |  0.0798731820 |  0.0654589398
     5 |      200 |  0.0753824471 |  0.0617657289
     6 |      240 |  0.0721291565 |  0.0616373592
     7 |      280 |  0.0691943055 |  0.0552590343
     8 |      320 |  0.0674683342 |  0.0519856068
     9 |      360 |  0.0650993282 |  0.0519856068
    10 |      400 |  0.0642941893 |  0.0519856068
    11 |      440 |  0.0623436389 |  0.0519856068
    12 |      480 |  0.0619115736 |  0.0519856068
    13 |      520 |  0.0615525175 |  0.0519856068
    14 |      560 |  0.0609960613 |  0.0519856068
    15 |      600 |  0.0607362018 |  0.0519856068
    16 |      640 |  0.0600126428 |  0.0519856068
    17 |      680 |  0.0594621239 |  0.0519856068
    18 |      720 |  0.0592126247 |  0.0519856068
    19

In [7]:
print("Evaluating best feature subset found by SGA...\n")

sga_best_mask = sga_result.X.astype(bool)
selected_features_count = int(sga_best_mask.sum())

start_time = time.time()
sga_final_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
sga_final_model.fit(X_train[:, sga_best_mask], y_train)
sga_final_train_time = time.time() - start_time

sga_predictions = sga_final_model.predict(X_test[:, sga_best_mask])
sga_accuracy = accuracy_score(y_test, sga_predictions)
sga_f1 = f1_score(y_test, sga_predictions)

print("FINAL RESULTS: Random Forest vs SGA")
print(f"{'Metric':<28}{'All Features':<18}{'SGA-Selected':<18}")
print(f"{'Accuracy':<28}{baseline_accuracy:<18.4f}{sga_accuracy:<18.4f}")
print(f"{'F1-score':<28}{baseline_f1:<18.4f}{sga_f1:<18.4f}")
print(f"{'Number of Features':<28}{n_features:<18}{selected_features_count:<18}")

reduction_pct = (1 - selected_features_count / n_features) * 100
print(f"{'Feature Reduction %':<28}{'-':<18}{reduction_pct:<18.1f}")
print(f"{'Model Training Time (s)':<28}{baseline_train_time:<18.2f}{sga_final_train_time:<18.2f}")

Evaluating best feature subset found by SGA...

FINAL RESULTS: Random Forest vs SGA
Metric                      All Features      SGA-Selected      
Accuracy                    0.9665            0.9680            
F1-score                    0.8690            0.8772            
Number of Features          1558              793               
Feature Reduction %         -                 49.1              
Model Training Time (s)     1.52              0.58              
